In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"



In [4]:
import torch
torch.set_num_threads(1)
print(torch.tensor([1.0, 2.0, 3.0]))

tensor([1., 2., 3.])


In [5]:
import pandas as pd
import cv2
df = pd.read_parquet("keypoints.parquet")
print(df.shape)
print(df["action"].value_counts())
print(df["clip_id"].nunique())
print(df["actor"].nunique())
print(df.columns)

(1654587, 10)
action
flat_service     443652
forehand_flat    422070
backhand         404613
smash            384252
Name: count, dtype: int64
660
55
Index(['clip_id', 'actor', 'action', 'seq', 'frame_idx', 'landmark_id', 'x',
       'y', 'z', 'visibility'],
      dtype='object')


In [6]:
df.head(5)

,clip_id,actor,action,seq,frame_idx,landmark_id,x,y,z,visibility
0,p10_backhand_s1,p10,backhand,1,0,0,0.530945,0.228937,-0.305087,0.999949
1,p10_backhand_s1,p10,backhand,1,0,1,0.537062,0.214805,-0.293559,0.999881
2,p10_backhand_s1,p10,backhand,1,0,2,0.540934,0.214909,-0.293701,0.999860
3,p10_backhand_s1,p10,backhand,1,0,3,0.544593,0.215270,-0.293769,0.999856
4,p10_backhand_s1,p10,backhand,1,0,4,0.525245,0.214773,-0.292146,0.999874


## Engineering Skill Targets
*(see ML_AVIATION_JOURNEY.md -> Running Practices -> README skill-check)*

1. **Problem framing** -- input: MediaPipe pose keypoints extracted from THETIS RGB video clips. Target: which of 4 strokes (`backhand`, `flat_service`, `forehand_flat`, `smash`). Baseline: logistic regression floor.
2. **Data understanding** -- done above: inspected raw video properties, confirmed keypoints visually by overlaying them back on frames (`visualize_clip.py`), understood the visibility field and the two kinds of missingness.
3. **Evaluation rigor** -- subject-based (`actor`) group split, not clip-based -- a subject's clips can't appear in both train and test. Evaluated across 5 different random splits, not just one (same rigor habit as NTSB's 5-splits-x-5-seeds table).
4. **Under-the-hood understanding** -- MediaPipe's pose model: used, not built (pretrained, frozen, inference-only). The feature engineering and the 3 classifiers below: ours, trained from scratch on our own labels.
5. **Build vs. use** -- pose detection = off-the-shelf (MediaPipe). Phase-aggregated features + trend deltas = designed here. Model comparison (logistic regression -> random forest -> XGBoost) = ours.
6. **Integration** -- this notebook is one step in the pipeline: raw video -> `extract_keypoints.py` (script, batch job) -> this notebook (features + models) -> later, Module D wraps the winning model in a coaching agent.

---

## Design decisions, so future-me remembers why

- **Phase-based aggregation, not whole-clip averaging.** Each clip's frames are split into 3 equal phases (early/mid/late) and each raw signal is averaged within each phase. A whole-clip average would erase the temporal shape of the swing entirely -- sequence pattern matters more than raw min/max/mean for distinguishing strokes. This is a middle ground: cheaper than a full sequence model, but not blind to order.
- **Explicit trend (late - early) features on top.** Trees split on one feature at a time and can't directly ask "is late bigger than early" in a single step -- handing the delta over directly saves the model from having to reconstruct it across several splits, which matters with only ~660 clips.
- **Low-visibility landmarks (< 0.5) are treated as missing**, not trusted, for any signal that depends on them.
- **Wrist speed is normalized by shoulder width** (a rough body-scale reference) and expressed per *frame*, not per second -- fps isn't saved per-clip in `keypoints.parquet`, and THETIS's Kinect capture is a consistent setup, so this is a reasonable simplification, flagged here in case results ever look fps-sensitive.
- **3-tier model comparison**, same escalation pattern as NTSB: logistic regression (floor baseline) -> random forest (solid default) -> XGBoost (the new one, likely best, worth learning specifically because it's the modern tabular-data standard).

## 2. Feature engineering

Turns the long-format keypoints table (one row per landmark per frame) into one row per clip -- the fixed-size shape a classifier needs.

In [7]:
import math
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed/importable -- that tier will be skipped.")

In [8]:
LEFT_SHOULDER, RIGHT_SHOULDER = 11, 12
LEFT_ELBOW, RIGHT_ELBOW = 13, 14
LEFT_WRIST, RIGHT_WRIST = 15, 16
LEFT_HIP, RIGHT_HIP = 23, 24
LEFT_KNEE, RIGHT_KNEE = 25, 26
LEFT_ANKLE, RIGHT_ANKLE = 27, 28

N_PHASES = 3
MIN_VISIBILITY = 0.5


def _get(frame, landmark_id, min_visibility):
    """Return (x, y) for a landmark in a frame dict, or None if missing/low-visibility."""
    row = frame.get(landmark_id)
    if row is None or row["visibility"] < min_visibility:
        return None
    return row["x"], row["y"]


def _angle(a, b, c):
    """Angle at point b, given three (x, y) points. NaN if any point missing."""
    if a is None or b is None or c is None:
        return np.nan
    v1 = (a[0] - b[0], a[1] - b[1])
    v2 = (c[0] - b[0], c[1] - b[1])
    n1, n2 = math.hypot(*v1), math.hypot(*v2)
    if n1 == 0 or n2 == 0:
        return np.nan
    cos_angle = (v1[0] * v2[0] + v1[1] * v2[1]) / (n1 * n2)
    cos_angle = max(-1.0, min(1.0, cos_angle))
    return math.degrees(math.acos(cos_angle))


def _dist(a, b):
    if a is None or b is None:
        return np.nan
    return math.hypot(a[0] - b[0], a[1] - b[1])

In [9]:
def clip_to_frames(clip_df, min_visibility):
    """Long-format rows for one clip -> {frame_idx: {landmark_id: {x,y,visibility}}}, sorted."""
    frames = {}
    for frame_idx, group in clip_df.groupby("frame_idx"):
        frames[frame_idx] = {
            int(r.landmark_id): {"x": r.x, "y": r.y, "visibility": r.visibility}
            for r in group.itertuples()
        }
    return dict(sorted(frames.items()))


def compute_raw_signals(frames, min_visibility):
    """Per-frame body signals for one clip, in frame order."""
    records = []
    prev_wrist = {"left": None, "right": None}
    prev_frame_idx = None

    for frame_idx, lm in frames.items():
        ls = _get(lm, LEFT_SHOULDER, min_visibility)
        rs = _get(lm, RIGHT_SHOULDER, min_visibility)
        le = _get(lm, LEFT_ELBOW, min_visibility)
        re = _get(lm, RIGHT_ELBOW, min_visibility)
        lw = _get(lm, LEFT_WRIST, min_visibility)
        rw = _get(lm, RIGHT_WRIST, min_visibility)
        lh = _get(lm, LEFT_HIP, min_visibility)
        rh = _get(lm, RIGHT_HIP, min_visibility)
        lk = _get(lm, LEFT_KNEE, min_visibility)
        rk = _get(lm, RIGHT_KNEE, min_visibility)
        la = _get(lm, LEFT_ANKLE, min_visibility)
        ra = _get(lm, RIGHT_ANKLE, min_visibility)

        shoulder_width = _dist(ls, rs)

        def speed(prev, cur):
            if prev is None or cur is None or not shoulder_width or np.isnan(shoulder_width):
                return np.nan
            gap = frame_idx - prev_frame_idx if prev_frame_idx is not None else 1
            gap = max(gap, 1)
            return _dist(prev, cur) / gap / shoulder_width

        rec = {
            "frame_idx": frame_idx,
            "left_elbow_angle": _angle(ls, le, lw),
            "right_elbow_angle": _angle(rs, re, rw),
            "left_knee_angle": _angle(lh, lk, la),
            "right_knee_angle": _angle(rh, rk, ra),
            "left_wrist_rel_height": (ls[1] - lw[1]) if ls and lw else np.nan,
            "right_wrist_rel_height": (rs[1] - rw[1]) if rs and rw else np.nan,
            "torso_rotation_deg": (
                math.degrees(math.atan2(rs[1] - ls[1], rs[0] - ls[0])) if ls and rs else np.nan
            ),
            "left_wrist_speed": speed(prev_wrist["left"], lw),
            "right_wrist_speed": speed(prev_wrist["right"], rw),
        }
        records.append(rec)
        prev_wrist = {"left": lw, "right": rw}
        prev_frame_idx = frame_idx

    return pd.DataFrame(records)

In [10]:
SIGNAL_COLUMNS = [
    "left_elbow_angle", "right_elbow_angle",
    "left_knee_angle", "right_knee_angle",
    "left_wrist_rel_height", "right_wrist_rel_height",
    "torso_rotation_deg",
    "left_wrist_speed", "right_wrist_speed",
]

TREND_SIGNALS = [
    "left_elbow_angle", "right_elbow_angle",
    "left_wrist_rel_height", "right_wrist_rel_height",
    "left_wrist_speed", "right_wrist_speed",
]


def phase_aggregate(signals_df):
    """One clip's per-frame signals -> one row of phase-aggregated + trend features."""
    n = len(signals_df)
    if n == 0:
        return {}

    phase_edges = np.linspace(0, n, N_PHASES + 1, dtype=int)
    phase_names = ["early", "mid", "late"]
    out = {}

    phase_means = {}
    for col in SIGNAL_COLUMNS:
        means = []
        for i, name in enumerate(phase_names):
            chunk = signals_df[col].iloc[phase_edges[i]:phase_edges[i + 1]]
            m = chunk.mean(skipna=True)
            out[f"{col}__{name}"] = m
            means.append(m)
        phase_means[col] = means

    for col in TREND_SIGNALS:
        early, _, late = phase_means[col]
        out[f"{col}__trend"] = (
            late - early if not (np.isnan(early) or np.isnan(late)) else np.nan
        )

    return out


def build_feature_table(kp_df, min_visibility):
    rows = []
    meta_cols = kp_df[["clip_id", "actor", "action", "seq"]].drop_duplicates("clip_id")
    meta_lookup = meta_cols.set_index("clip_id").to_dict("index")

    for clip_id, clip_df in kp_df.groupby("clip_id"):
        frames = clip_to_frames(clip_df, min_visibility)
        signals_df = compute_raw_signals(frames, min_visibility)
        features = phase_aggregate(signals_df)
        meta = meta_lookup[clip_id]
        rows.append({"clip_id": clip_id, **meta, "n_frames": len(frames), **features})

    return pd.DataFrame(rows)

In [11]:
feature_df = build_feature_table(df, MIN_VISIBILITY)

n_missing = feature_df.isna().any(axis=1).sum()
print(f"Built {len(feature_df)} clip rows, {feature_df.shape[1]} columns "
      f"({n_missing} rows have at least one NaN feature -- expected for clips "
      f"with detection gaps, imputed at training time).")

feature_df.to_parquet("features.parquet", index=False)
feature_df.head()

Built 660 clip rows, 38 columns (107 rows have at least one NaN feature -- expected for clips with detection gaps, imputed at training time).


,clip_id,actor,action,seq,n_frames,left_elbow_angle__early,left_elbow_angle__mid,left_elbow_angle__late,right_elbow_angle__early,right_elbow_angle__mid,...,left_wrist_speed__late,right_wrist_speed__early,right_wrist_speed__mid,right_wrist_speed__late,left_elbow_angle__trend,right_elbow_angle__trend,left_wrist_rel_height__trend,right_wrist_rel_height__trend,left_wrist_speed__trend,right_wrist_speed__trend
0,p10_backhand_s1,p10,backhand,1,78,98.474001,93.246376,141.301121,152.842834,130.453368,...,0.231015,0.168901,0.053075,0.045653,42.827120,2.202793,0.060424,-0.033653,-0.025938,-0.123248
1,p10_backhand_s2,p10,backhand,2,73,105.039974,117.617637,132.604655,145.680838,125.211773,...,0.175968,0.261293,0.068683,0.026487,27.564681,-16.531516,0.057267,-0.027266,-0.029325,-0.234805
2,p10_backhand_s3,p10,backhand,3,66,70.221545,127.922165,150.939571,116.214722,118.668403,...,0.222404,0.350744,0.062737,0.019449,80.718026,16.080792,-0.000865,-0.062562,-0.175336,-0.331295
3,p10_foreflat_s1,p10,forehand_flat,1,89,147.900331,142.847274,108.854906,140.782029,120.375952,...,0.141313,0.916796,0.746775,0.097182,-39.045425,-6.569662,0.090881,0.024375,-0.558700,-0.819615
4,p10_foreflat_s2,p10,forehand_flat,2,88,130.129869,132.310832,113.456106,139.861529,148.774662,...,0.125340,0.083997,0.117892,0.063735,-16.673763,13.499604,0.078120,0.014597,0.007498,-0.020262


## 3. Model comparison

logistic regression (floor baseline) -> random forest (solid default) -> XGBoost (likely best).

Split by `actor` (subject), not by clip -- a subject's clips can't appear in both train and test. Evaluated across 5 different random group-splits, not just one.

In [12]:
NON_FEATURE_COLS = {"clip_id", "actor", "action", "seq", "n_frames"}

feature_cols = [c for c in feature_df.columns if c not in NON_FEATURE_COLS]
X = feature_df[feature_cols]
y = feature_df["action"]
groups = feature_df["actor"]

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print(f"{len(X)} clips, {X.shape[1]} features, {len(encoder.classes_)} classes: "
      f"{list(encoder.classes_)}")
print(f"{groups.nunique()} unique subjects.")

660 clips, 33 features, 4 classes: ['backhand', 'flat_service', 'forehand_flat', 'smash']
55 unique subjects.


In [13]:
def make_models():
    """One dict entry per tier -- easy to add a 4th later without touching the rest."""
    models = {
        "logistic_regression": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000)),
        ]),
        "random_forest": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(n_estimators=300, random_state=0)),
        ]),
    }
    if HAS_XGB:
        models["xgboost"] = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=300, max_depth=4, learning_rate=0.1,
                eval_metric="mlogloss", random_state=0,
            )),
        ])
    return models


def run_one_split(models, X, y_encoded, groups, seed):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y_encoded, groups))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    results = {}
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        results[name] = {
            "accuracy": accuracy_score(y_test, preds),
            "macro_f1": f1_score(y_test, preds, average="macro"),
        }
    return results, (X_test, y_test)


def multi_seed_eval(models, X, y_encoded, groups, n_seeds=5):
    """Same rigor pattern as NTSB -- one split can mislead, several can't as easily."""
    all_results = {name: {"accuracy": [], "macro_f1": []} for name in models}
    last_test_set = None

    for seed in range(n_seeds):
        results, test_set = run_one_split(models, X, y_encoded, groups, seed)
        last_test_set = test_set
        for name, metrics in results.items():
            all_results[name]["accuracy"].append(metrics["accuracy"])
            all_results[name]["macro_f1"].append(metrics["macro_f1"])

    return all_results, last_test_set

In [14]:
models = make_models()
all_results, last_test_set = multi_seed_eval(models, X, y_encoded, groups, n_seeds=5)

print("=== Model comparison across 5 group-splits ===")
for name, metrics in all_results.items():
    acc = np.array(metrics["accuracy"])
    f1 = np.array(metrics["macro_f1"])
    print(f"{name:20s}  accuracy {acc.mean():.3f} +/- {acc.std():.3f}"
          f"   macro-F1 {f1.mean():.3f} +/- {f1.std():.3f}"
          f"   (range {acc.min():.3f}-{acc.max():.3f})")

=== Model comparison across 5 group-splits ===
logistic_regression   accuracy 0.663 +/- 0.041   macro-F1 0.662 +/- 0.039   (range 0.583-0.696)
random_forest         accuracy 0.662 +/- 0.048   macro-F1 0.661 +/- 0.046   (range 0.601-0.714)
xgboost               accuracy 0.671 +/- 0.023   macro-F1 0.670 +/- 0.020   (range 0.637-0.702)


In [15]:
for name in ("random_forest", "xgboost"):
    pipe = models.get(name)
    if pipe is None:
        continue
    clf = pipe.named_steps["clf"]
    importances = getattr(clf, "feature_importances_", None)
    if importances is None:
        continue
    order = np.argsort(importances)[::-1][:10]
    print(f"\nTop 10 features -- {name}:")
    for i in order:
        print(f"  {feature_cols[i]:35s} {importances[i]:.4f}")


Top 10 features -- random_forest:
  right_wrist_rel_height__mid         0.0780
  left_elbow_angle__late              0.0568
  left_elbow_angle__early             0.0559
  left_wrist_rel_height__late         0.0557
  left_elbow_angle__trend             0.0453
  left_wrist_rel_height__mid          0.0437
  torso_rotation_deg__late            0.0412
  right_elbow_angle__early            0.0394
  left_wrist_rel_height__trend        0.0379
  torso_rotation_deg__mid             0.0375

Top 10 features -- xgboost:
  left_elbow_angle__early             0.1155
  left_elbow_angle__late              0.0804
  right_wrist_rel_height__mid         0.0560
  left_wrist_rel_height__trend        0.0536
  left_elbow_angle__trend             0.0416
  left_wrist_rel_height__mid          0.0399
  right_knee_angle__mid               0.0379
  right_elbow_angle__early            0.0358
  right_elbow_angle__late             0.0350
  torso_rotation_deg__late            0.0344


In [16]:
X_test, y_test = last_test_set
rf = models["random_forest"]
preds = rf.predict(X_test)

print("Confusion matrix (random forest, last split) -- rows=true, cols=predicted:")
display(pd.DataFrame(
    confusion_matrix(y_test, preds),
    index=encoder.classes_, columns=encoder.classes_,
))

print("\nFull classification report (random forest, last split):")
print(classification_report(y_test, preds, target_names=encoder.classes_))

Confusion matrix (random forest, last split) -- rows=true, cols=predicted:


,backhand,flat_service,forehand_flat,smash
backhand,29,4,7,2
flat_service,7,24,0,11
forehand_flat,2,11,24,5
smash,1,16,1,24



Full classification report (random forest, last split):
               precision    recall  f1-score   support

     backhand       0.74      0.69      0.72        42
 flat_service       0.44      0.57      0.49        42
forehand_flat       0.75      0.57      0.65        42
        smash       0.57      0.57      0.57        42

     accuracy                           0.60       168
    macro avg       0.63      0.60      0.61       168
 weighted avg       0.63      0.60      0.61       168



## 4. 1D-CNN on the raw sequence (not phase-aggregated)

Testing whether keeping the full frame-by-frame order fixes the serve/smash
confusion the tabular models couldn't (see confusion matrix above).

Design, per the CNN-vs-LSTM discussion above:
- Each clip's raw per-frame signals (same 9 signals as the feature engineering
  step, just *not* phase-aggregated) are resampled to a fixed length so every
  clip is directly comparable, using linear interpolation.
- The architecture **deliberately avoids global pooling** at the end -- it
  flattens the conv output while still keeping the time axis, so *where*
  along the clip a pattern fired isn't thrown away. That's the whole point:
  a CNN that global-pools away position couldn't represent "toss happens
  before the swing" vs. "no toss" -- this one can.
- Same subject-based group split and per-channel standardization (fit on
  train only, to avoid leaking test statistics).

In [17]:
import torch
import torch.nn as nn

torch.manual_seed(0)

In [18]:
def resample_signals(signals_df, target_len=64):
    """One clip's per-frame signal DataFrame -> fixed-length (channels, target_len) array."""
    n = len(signals_df)
    if n < 2:
        return None
    old_idx = np.linspace(0, 1, n)
    new_idx = np.linspace(0, 1, target_len)
    channels = []
    for col in SIGNAL_COLUMNS:
        s = signals_df[col].astype(float).interpolate(limit_direction="both")
        s = s.fillna(0.0)  # only happens if a signal is NaN for the whole clip
        channels.append(np.interp(new_idx, old_idx, s.values))
    return np.stack(channels, axis=0)  # (n_channels, target_len)


def build_sequence_dataset(kp_df, min_visibility, target_len=64):
    X_list, y_list, actor_list = [], [], []
    meta_cols = kp_df[["clip_id", "actor", "action"]].drop_duplicates("clip_id")
    meta_lookup = meta_cols.set_index("clip_id").to_dict("index")

    for clip_id, clip_df in kp_df.groupby("clip_id"):
        frames = clip_to_frames(clip_df, min_visibility)
        signals_df = compute_raw_signals(frames, min_visibility)
        seq = resample_signals(signals_df, target_len)
        if seq is None:
            continue
        meta = meta_lookup[clip_id]
        X_list.append(seq)
        y_list.append(meta["action"])
        actor_list.append(meta["actor"])

    return np.stack(X_list), np.array(y_list), np.array(actor_list)


SEQ_LEN = 64
X_seq, y_seq, groups_seq = build_sequence_dataset(df, MIN_VISIBILITY, target_len=SEQ_LEN)
y_seq_encoded = encoder.transform(y_seq)  # reuse the same label encoder as the tabular models
print(f"X_seq shape: {X_seq.shape}  (clips, channels, timesteps)")

X_seq shape: (660, 9, 64)  (clips, channels, timesteps)


In [19]:
class StrokeCNN(nn.Module):
    """
    Two conv layers + moderate (not global) pooling, so the flatten step
    still carries *some* notion of early/mid/late position into the dense
    layers -- deliberately not collapsed to a single vector per channel.
    """
    def __init__(self, n_channels, seq_len, n_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(n_channels, 32, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)
        reduced_len = seq_len // 4  # two stride-2 pools
        self.fc1 = nn.Linear(64 * reduced_len, 64)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.flatten(1)          # keeps time-bucket structure, not global-pooled away
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [20]:
import time

def train_and_eval_cnn(X_train, y_train, X_test, y_test, n_classes, epochs=60, lr=1e-3, seed=0):
    print("    building model...", flush=True)
    torch.manual_seed(seed)
    model = StrokeCNN(X_train.shape[1], X_train.shape[2], n_classes)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()

    print("    converting data to tensors...", flush=True)
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    print(f"    tensors ready ({X_train_t.shape[0]} train, {X_test_t.shape[0]} test) -- starting {epochs} epochs", flush=True)
    epoch_start = time.time()
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(X_train_t), y_train_t)
        loss.backward()
        opt.step()
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"    epoch {epoch+1}/{epochs}  loss={loss.item():.4f}  ({time.time()-epoch_start:.1f}s elapsed)", flush=True)

    print("    training done, evaluating on test set...", flush=True)
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).argmax(dim=1).numpy()
    print("    done.", flush=True)
    return preds, model


def run_cnn_split(X_seq, y_encoded, groups, seed):
    print(f"  splitting data for seed {seed}...", flush=True)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    train_idx, test_idx = next(splitter.split(X_seq, y_encoded, groups))
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
    print(f"  split done: {len(train_idx)} train, {len(test_idx)} test. standardizing...", flush=True)

    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    print("  calling train_and_eval_cnn...", flush=True)
    preds, model = train_and_eval_cnn(X_train, y_train, X_test, y_test,
                                       n_classes=len(encoder.classes_), seed=seed)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    return acc, f1, (y_test, preds)

In [ ]:
cnn_accs, cnn_f1s = [], []
last_cnn_result = None
overall_start = time.time()

for seed in range(3):
    print(f"=== seed {seed} starting ===", flush=True)
    seed_start = time.time()
    acc, f1, result = run_cnn_split(X_seq, y_seq_encoded, groups_seq, seed)
    cnn_accs.append(acc)
    cnn_f1s.append(f1)
    last_cnn_result = result
    print(f"=== seed {seed} done: accuracy {acc:.3f}  macro-F1 {f1:.3f}  ({time.time()-seed_start:.1f}s) ===\n", flush=True)

print(f"Total time: {time.time() - overall_start:.1f}s")
cnn_accs, cnn_f1s = np.array(cnn_accs), np.array(cnn_f1s)
print(f"\n1D-CNN over {len(cnn_accs)} splits: accuracy {cnn_accs.mean():.3f} +/- {cnn_accs.std():.3f}"
      f"   macro-F1 {cnn_f1s.mean():.3f} +/- {cnn_f1s.std():.3f}")

=== seed 0 starting ===
  splitting data for seed 0...
  split done: 492 train, 168 test. standardizing...
  calling train_and_eval_cnn...
    building model...
    converting data to tensors...


In [ ]:
# Confusion matrix for the CNN -- specifically check whether serve/smash improved
y_test_last, preds_last = last_cnn_result
print("Confusion matrix (1D-CNN, last split) -- rows=true, cols=predicted:")
display(pd.DataFrame(
    confusion_matrix(y_test_last, preds_last),
    index=encoder.classes_, columns=encoder.classes_,
))
print("\nFull classification report (1D-CNN, last split):")
print(classification_report(y_test_last, preds_last, target_names=encoder.classes_))